![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [1]:
# Initialize QuantBook
qb = QuantBook()

# Example: Set Benchmark and add technology stocks
qb.SetBenchmark("SPY")
tech_stocks = [
    "AAPL", "MSFT"]
#     , "GOOGL", "AMZN", "NVDA",
#     "META", "TSLA", "ADBE", "CRM", "INTC",
#     "AMD", "ORCL", "CSCO", "QCOM", "IBM",
#     "TXN", "AVGO", "SHOP", "UBER", "PYPL"
# ]

history_data = {}
options_data = {}
for stock in tech_stocks:
    symbol = qb.AddEquity(stock, Resolution.Daily).Symbol
    qb.AddOption(symbol)  # Add options for the stock
    start_date = pd.Timestamp("2024-01-01")
    end_date = pd.Timestamp("2024-12-01")
    df = qb.History([symbol], start_date, end_date, Resolution.Daily).reset_index()
    df.set_index("time", inplace=True)  # Ensure proper indexing for time alignment
    df["normalized_close"] = (df["close"] - df["close"].mean()) / df["close"].std()  # Normalize the close prices
    history_data[symbol.Value] = df  # Store data using string representation of the Symbol
    options_history = qb.OptionHistory(symbol, start_date, end_date, Resolution.Daily)
    options_data[symbol.Value] = options_history
    print(f"Loaded history and options for {stock}")

    

In [23]:
# Adjust pandas settings to prevent truncation
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Adjust to terminal width
pd.set_option('display.max_rows', None)  # Show all rows (if needed)
# print(df.head())

# print(dir(options_data['AAPL']))
# print(options_data['AAPL'].DataFrame)
columns_dataframe = pd.DataFrame(options_data['AAPL'].DataFrame)
# print("Columns DataFrame head",'\n', columns_dataframe.head())
print("Columns of the dataframe", columns_dataframe.columns)
print("Index of the dataframe", columns_dataframe.index)



In [2]:
### Exploration code

import inspect
attributes_options_history = dir(options_history)
print(attributes_options_history)

att_qb = dir(qb)
print(att_qb)

sig = inspect.signature(qb.AddOption)
for param in sig.parameters.values():
    print(param.name)
    
attris = options_history.get_strikes()
print(attris)

In [ ]:
# Example: Load historical data
symbol = qb.AddEquity("SPY", Resolution.Daily).Symbol
qb.AddOption(symbol)  # Add options for SPY
start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2024-12-01")
data = qb.History([symbol], start_date, end_date, Resolution.Daily).reset_index()
data.set_index("time", inplace=True)  # Ensure proper indexing for time alignment
data["normalized_close"] = (data["close"] - data["close"].mean()) / data["close"].std()  # Normalize the close prices
options_history = qb.OptionHistory(symbol, start_date, end_date, Resolution.Daily)
options_data[symbol.Value] = options_history

# Check correlations between stocks and keep the 10 strongest correlations
import itertools
import pandas as pd

stock_symbols = list(history_data.keys())  # Use string keys for easier handling
correlation_results = []

for pair in itertools.combinations(stock_symbols, 2):
    stock1, stock2 = pair
    df1 = history_data[stock1]["normalized_close"].dropna()
    df2 = history_data[stock2]["normalized_close"].dropna()
    if not df1.empty and not df2.empty:
        combined = pd.DataFrame({"stock1": df1, "stock2": df2}).dropna()
        correlation = combined["stock1"].corr(combined["stock2"])
        correlation_results.append((stock1, stock2, correlation))

# Sort correlations by absolute value and take the top 10
strongest_correlations = sorted(correlation_results, key=lambda x: abs(x[2]), reverse=True)[:10]

print("Top 10 strongest correlations between stocks:")
for pair in strongest_correlations:
    print(f"{pair[0]} and {pair[1]}: Correlation = {pair[2]:.2f}")

In [ ]:
# Example: Calculate and plot normalized spreads for top correlated pairs
import matplotlib.pyplot as plt

for pair in strongest_correlations:
    stock1, stock2, _ = pair
    df1 = history_data[stock1]["normalized_close"].dropna()
    df2 = history_data[stock2]["normalized_close"].dropna()
    spread = df1 - df2

    plt.figure(figsize=(10, 6))
    plt.plot(spread.index, spread, label=f"Normalized Spread: {stock1} - {stock2}")
    plt.title(f"Normalized Spread between {stock1} and {stock2}")
    plt.xlabel("Date")
    plt.ylabel("Normalized Spread")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

for pair in strongest_correlations:
    stock1, stock2, _ = pair
    iv1_data = [contract.ImpliedVolatility for slice in options_data[stock1] if hasattr(slice, "OptionChains") for chain in slice.OptionChains.values() for contract in chain.Contracts.values() if contract.ImpliedVolatility > 0]
    iv2_data = [contract.ImpliedVolatility for slice in options_data[stock2] if hasattr(slice, "OptionChains") for chain in slice.OptionChains.values() for contract in chain.Contracts.values() if contract.ImpliedVolatility > 0]

    if iv1_data and iv2_data:
        iv1 = pd.Series(iv1_data).mean()
        iv2 = pd.Series(iv2_data).mean()

        # Create a dual-axis plot for implied volatilities
        fig, ax1 = plt.subplots(figsize=(10, 6))
        ax2 = ax1.twinx()

        ax1.plot(iv1_data, label=f"{stock1} IV", color='blue')
        ax2.plot(iv2_data, label=f"{stock2} IV", color='orange')

        ax1.set_xlabel("Date")
        ax1.set_ylabel(f"{stock1} Implied Volatility", color='blue')
        ax2.set_ylabel(f"{stock2} Implied Volatility", color='orange')
        plt.title(f"Implied Volatility of {stock1} and {stock2}")
        plt.legend(loc="upper left")
        plt.grid()
        plt.show()

        # Plot the normalized spread of implied volatilities
        iv1_norm = (pd.Series(iv1_data) - iv1) / pd.Series(iv1_data).std()
        iv2_norm = (pd.Series(iv2_data) - iv2) / pd.Series(iv2_data).std()
        spread_norm = iv1_norm - iv2_norm

        plt.figure(figsize=(10, 6))
        plt.plot(spread_norm, label=f"Normalized IV Spread: {stock1} - {stock2}")
        plt.title(f"Normalized Implied Volatility Spread between {stock1} and {stock2}")
        plt.xlabel("Date")
        plt.ylabel("Normalized Spread")
        plt.legend()
        plt.grid()
        plt.show()

In [ ]:
# Add the option chain for SPY

option_chain = qb.OptionChain("SPY")

# Access the contracts from the option chain
contracts = option_chain.Contracts

# Check the number of contracts
print(f"Number of contracts: {len(contracts)}")

# Loop through and display the contracts
for symbol, contract in contracts.items():
    print(f"Symbol: {symbol}, Strike: {contract.Strike}, Expiry: {contract.Expiry}, Right: {contract.Right}")


In [ ]:
from datetime import timedelta

# Define a filter for the option chain
def universe_filter(option_chain):
    return [
        contract
        for contract in option_chain
        if contract.Expiry < qb.Time + timedelta(days=60) and
           contract.Strike > 400 and
           contract.Right == OptionRight.Call
    ]

# Apply the filter
filtered_options = universe_filter(option_chain)

# Print filtered contracts
print(f"Filtered contracts: {len(filtered_options)}")
for contract in filtered_options:
    print(contract)

In [ ]:
# Get historical data for a specific option contract
if filtered_options:
    selected_contract = filtered_options[0]  # Pick the first contract
    history = qb.History([selected_contract.Symbol], 30, Resolution.Minute)
    print(history.head())

In [ ]:
import matplotlib.pyplot as plt

# Plot the historical close prices
if not history.empty:
    history["close"].plot(title="Option Contract Price")
    plt.show()